<a href="https://colab.research.google.com/github/PepeFederico/Progetto_Mercati_Finanziari/blob/main/rendimenti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# Import delle librerie
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from sklearn.model_selection import train_test_split

In [35]:
# 1. Definiamo i 7 titoli
tickers = ['NVDA', 'GOOGL', 'AMZN', 'BRK-B', 'LLY', 'WMT', 'XOM']

# 2. Scarichiamo i dati degli ultimi 2 anni
print("Scaricamento dati in corso...")
data = yf.download(tickers, period='10y', interval='1d')['Close']

#   Trasforma i prezzi in logaritmici
log_data = np.log(data)

# --- 3. CREAZIONE DEI TASSI CON FINESTRE SCORREVOLI ---

# A. Giornaliero (finestra = 1 giorno)
returns_1d = log_data.diff(1)

# B. Settimanale rolling (finestra = 5 giorni lavorativi)
returns_5d = log_data.diff(5)

# B.1. Settimanale (passo = 1 settimana) -> Prendiamo il dato di fine settimana ('W')
data_weekly = data.resample('W').last()
returns_1w = np.log(data_weekly).diff(1)

# C. Mensile rolling (finestra = 21 giorni lavorativi)
returns_21d = log_data.diff(21)

# D. Trimestrale rolling (finestra = 63 giorni lavorativi)
returns_63d = log_data.diff(63)

# 4. Rinominiamo le colonne per mantenere ordine
returns_1d.columns    = [f"{col}_Ret_1D" for col in returns_1d.columns]
returns_5d.columns    = [f"{col}_Ret_1W" for col in returns_5d.columns]
returns_1w.columns    = [f"{col}_Ret_1W" for col in returns_1w.columns]
returns_21d.columns   = [f"{col}_Ret_1M" for col in returns_21d.columns]
returns_63d.columns   = [f"{col}_Ret_3M" for col in returns_63d.columns]

# 5. Uniamo tutto orizzontalmente
dataset_finale = pd.concat([returns_1d, returns_5d, returns_1w, returns_21d, returns_63d], axis=1)

returns_1w_riallineato = returns_1w.reindex(returns_1d.index).ffill()

dataset_rendimenti_giornalieri                    = pd.concat([returns_1d], axis=1)
dataset_rendimenti_settimanali_sliding_window     = pd.concat([returns_5d], axis=1)
dataset_rendimenti_settimanali_tumbling_window    = pd.concat([returns_1w_riallineato], axis=1)
dataset_rendimenti_settimanali_mensili            = pd.concat([returns_21d], axis=1)
dataset_rendimenti_settimanali_trimestrali        = pd.concat([returns_63d], axis=1)

# 6. Pulizia: eliminiamo le prime 63 righe che conterranno dei "NaN"
#dataset_finale = dataset_finale.dropna() # Lasciato commentato come da tua richiesta

# 7. Salvataggio in CSV
csv_filename = "rendimenti_multi_timeframe.csv"
dataset_finale.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")

csv_filename = "rendimenti_giornalieri.csv"
dataset_rendimenti_giornalieri.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")

csv_filename = "rendimenti_settimanali_tumbling.csv"
dataset_rendimenti_settimanali_tumbling_window.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")

csv_filename = "rendimenti_settimanali_sliding.csv"
dataset_rendimenti_settimanali_sliding_window.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")

csv_filename = "rendimenti_mensili.csv"
dataset_rendimenti_settimanali_mensili.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")

csv_filename = "rendimenti_trimestrali.csv"
dataset_rendimenti_settimanali_trimestrali.to_csv(csv_filename)
print(f"Dati estratti e salvati in: {csv_filename}")
print(f"Dimensioni del dataset: {dataset_finale.shape}")


# --- 8. CREAZIONE DEI GRAFICI PER COMPAGNIA E PERIODO ---
print("\nGenerazione dei grafici in corso...")

for ticker in tickers:
    # Identifichiamo le 4 colonne che appartengono a questa specifica azienda
    col_1d = f"{ticker}_Ret_1D"
    col_1w = f"{ticker}_Ret_1W"
    col_1m = f"{ticker}_Ret_1M"
    col_3m = f"{ticker}_Ret_3M"

    # Creiamo una "tela" con 4 sotto-grafici (2 righe, 2 colonne)
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 8))
    fig.suptitle(f"Andamento Storico dei Rendimenti - {ticker}", fontsize=16, fontweight='bold')

    # Grafico 1: Giornaliero (In alto a sinistra)
    dataset_finale[col_1d].plot(ax=axes[0, 0], color='blue', alpha=0.7)
    axes[0, 0].set_title('Giornaliero (1D)')
    axes[0, 0].axhline(0, color='black', linewidth=1, linestyle='--') # Linea dello zero
    axes[0, 0].grid(True, linestyle=':', alpha=0.6)

    # Grafico 2: Settimanale (In alto a destra)
    dataset_finale[col_1w].plot(ax=axes[0, 1], color='orange', alpha=0.7)
    axes[0, 1].set_title('Settimanale (1W)')
    axes[0, 1].axhline(0, color='black', linewidth=1, linestyle='--')
    axes[0, 1].grid(True, linestyle=':', alpha=0.6)

    # Grafico 3: Mensile (In basso a sinistra)
    dataset_finale[col_1m].plot(ax=axes[1, 0], color='green', alpha=0.7)
    axes[1, 0].set_title('Mensile (1M)')
    axes[1, 0].axhline(0, color='black', linewidth=1, linestyle='--')
    axes[1, 0].grid(True, linestyle=':', alpha=0.6)

    # Grafico 4: Trimestrale (In basso a destra)
    dataset_finale[col_3m].plot(ax=axes[1, 1], color='red', alpha=0.7)
    axes[1, 1].set_title('Trimestrale (3M)')
    axes[1, 1].axhline(0, color='black', linewidth=1, linestyle='--')
    axes[1, 1].grid(True, linestyle=':', alpha=0.6)

    # Miglioriamo la spaziatura tra i grafici
    plt.tight_layout()

    # Salviamo l'immagine sul computer
    plot_filename = f"grafico_{ticker}.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"Salvato grafico: {plot_filename}")

    # Se stai usando Jupyter/Colab e vuoi anche vederli a schermo subito, togli il # dalla riga qui sotto:
    # plt.show()

    # Chiudiamo la figura per non saturare la memoria del computer
    plt.close()

print("\nProcesso completato! Controlla la tua cartella per il CSV e le 7 immagini.")

Scaricamento dati in corso...


/tmp/ipykernel_11297/4056563534.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period='10y', interval='1d')['Close']
[*********************100%***********************]  7 of 7 completed


Dati estratti e salvati in: rendimenti_multi_timeframe.csv
Dimensioni del dataset: (3036, 35)
Dati estratti e salvati in: rendimenti_giornalieri.csv
Dimensioni del dataset: (3036, 35)
Dati estratti e salvati in: rendimenti_settimanali_tumbling.csv
Dimensioni del dataset: (3036, 35)
Dati estratti e salvati in: rendimenti_settimanali_sliding.csv
Dimensioni del dataset: (3036, 35)
Dati estratti e salvati in: rendimenti_mensili.csv
Dimensioni del dataset: (3036, 35)
Dati estratti e salvati in: rendimenti_trimestrali.csv
Dimensioni del dataset: (3036, 35)

Generazione dei grafici in corso...
Salvato grafico: grafico_NVDA.png
Salvato grafico: grafico_GOOGL.png
Salvato grafico: grafico_AMZN.png
Salvato grafico: grafico_BRK-B.png
Salvato grafico: grafico_LLY.png
Salvato grafico: grafico_WMT.png
Salvato grafico: grafico_XOM.png

Processo completato! Controlla la tua cartella per il CSV e le 7 immagini.


In [39]:
colonne_giornalieri = ['NVDA_Ret_1D', 'GOOGL_Ret_1D', 'BRK-B_Ret_1D', 'LLY_Ret_1D', 'WMT_Ret_1D', 'XOM_Ret_1D']
colonne_settimanali = ['NVDA_Ret_1W', 'GOOGL_Ret_1W', 'BRK-B_Ret_1W', 'LLY_Ret_1W', 'WMT_Ret_1W', 'XOM_Ret_1W']
colonne_trimestrali = ['NVDA_Ret_3M', 'GOOGL_Ret_3M', 'BRK-B_Ret_3M', 'LLY_Ret_3M', 'WMT_Ret_3M', 'XOM_Ret_3M']

df_giornalieri = dataset_rendimenti_giornalieri.drop(colonne_giornalieri, axis = 1)
df_settimanali = dataset_rendimenti_settimanali_sliding_window.drop(colonne_settimanali, axis = 1)
df_trimestrali = dataset_rendimenti_settimanali_trimestrali.drop(colonne_trimestrali, axis = 1)

#   Dataset pronti per modello
df_g_finale = df_giornalieri.loc['2016-09-15':]
df_s_finale = df_settimanali.loc['2016-09-15':]
df_w_finale = df_trimestrali.loc['2016-09-15':]

# --- STEP 1: DEFINIZIONE DEL TARGET ---
# Prendiamo il rendimento giornaliero e lo shiftiamo di -1.
# Significa che il valore di domani viene associato alla riga di oggi.
target_giornaliero = df_settimanali['AMZN_Ret_1W'].shift(-5)
y_finale = target_giornaliero.values


# --- STEP 4: CREAZIONE DELLE FINESTRE TEMPORALI (LOOKBACK 3D) ---
lookback_g = 20  # Ultime 20 giornate di borsa
lookback_s = 10  # Ultime 10 settimane sliding
lookback_w = 6   # Ultimi 6 trimestri sliding (impostato a 6 per combaciare con la rete)

def crea_finestre_3d(data, lookback):
    X_3d = []
    for i in range(lookback, len(data)):
        X_3d.append(data.iloc[i-lookback:i].values)
    return np.array(X_3d)

X_3d_g = crea_finestre_3d(df_g_finale, lookback_g)
X_3d_s = crea_finestre_3d(df_s_finale, lookback_s)
X_3d_w = crea_finestre_3d(df_w_finale, lookback_w)

# Sincronizzazione dell'inizio in base al lookback massimo (20)
max_lookback = max(lookback_g, lookback_s, lookback_w)

X_3d_g = X_3d_g[(max_lookback - lookback_g):]
X_3d_s = X_3d_s[(max_lookback - lookback_s):]
X_3d_w = X_3d_w[(max_lookback - lookback_w):]
y_target_sincro = y_finale[max_lookback:]


# --- STEP 5: SPLIT CRONOLOGICO (NO SHUFFLING) ---
total_samples = len(y_target_sincro)
split_test_idx = int(total_samples * 0.80)
split_val_idx = int(split_test_idx * 0.90)

# Split Ramo Giornaliero (Shape finale attesa: [Campioni, 20, 1])
X_train_g, X_val_g, X_test_g = X_3d_g[:split_val_idx], X_3d_g[split_val_idx:split_test_idx], X_3d_g[split_test_idx:]
# Split Ramo Settimanale (Shape finale attesa: [Campioni, 10, 1])
X_train_s, X_val_s, X_test_s = X_3d_s[:split_val_idx], X_3d_s[split_val_idx:split_test_idx], X_3d_s[split_test_idx:]
# Split Ramo Trimestrale (Shape finale attesa: [Campioni, 6, 1])
X_train_w, X_val_w, X_test_w = X_3d_w[:split_val_idx], X_3d_w[split_val_idx:split_test_idx], X_3d_w[split_test_idx:]

# Split Target
y_train, y_val, y_test = y_target_sincro[:split_val_idx], y_target_sincro[split_val_idx:split_test_idx], y_target_sincro[split_test_idx:]


In [40]:
#   Costruzione della rete

# 1° Ramo: Giornaliero
input_giornaliero = Input(shape=(20, 1), name = "Input_Giornaliero")
lstm_giornaliero = LSTM(64, return_sequences = False)(input_giornaliero)

# 2° Ramo: Settimanale
input_settimanale = Input(shape=(10, 1), name='Input_Settimanale')
lstm_settimanale = LSTM(32, return_sequences=False)(input_settimanale)

#3° Ramo: Trimestrale
input_trimestrale = Input(shape=(6, 1), name='Input_Trimestrale')
lstm_trimestrale = LSTM(16, return_sequences=False)(input_trimestrale)

# Concatenazione Livelli
concatenato = Concatenate()([lstm_giornaliero, lstm_settimanale, lstm_trimestrale])

# Passiamo il vettore fuso a degli strati densi per la decisione finale
denso_1 = Dense(32, activation='relu')(concatenato)
# Strato di output: 1 solo neurone con attivazione lineare (perché è una regressione del rendimento)
output_finale = Dense(1, activation='linear', name='Output_Predizione')(denso_1)

model = Model(inputs=[input_giornaliero, input_settimanale, input_trimestrale], outputs=output_finale)

# Compilazione
model.compile(optimizer='adam', loss='mse')

# Mostra la struttura della rete
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Giornaliero   │ (None, 20, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Settimanale   │ (None, 10, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Trimestrale   │ (None, 6, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ (None, 64)        │     16,896 │ Input_Giornalier… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ (None, 32)        │      4,352 │ Input_Settimanal… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_14 (LSTM)      │ (None, 16)        │      1,152 │ Input_Trimestral… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 112)       │          0 │ lstm_12[0][0],    │
│ (Concatenate)       │                   │            │ lstm_13[0][0],    │
│                     │                   │            │ lstm_14[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      3,616 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Predizione   │ (None, 1)         │         33 │ dense_4[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,049 (101.75 KB)

 Trainable params: 26,049 (101.75 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
#   Addestramento modello

history = model.fit(
    x=[X_train_g, X_train_s, X_train_w],
    y=y_train,
    validation_data=([X_val_g, X_val_s, X_val_w], y_val),
    epochs=50,
    batch_size=32
)

Epoch 1/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0019 - val_loss: 0.0012
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0019 - val_loss: 0.0012
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0019 - val_loss: 0.0012
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0019 - val_loss: 0.0012
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0019 - val_loss: 0.0011
Epoch 6/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0019 - val_loss: 0.0011
Epoch 7/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0019 - val_loss: 0.0011
Epoch 8/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0018 - val_loss: 0.0011
Epoch 9/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0018 - val_loss: 0.0012
Epoch 10/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0018 - val_loss: 0.0012
Epoch 11/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0018 - val_loss: 0.0012
Epoch 12/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0018 - v